In [ ]:
from google.colab import files
import os

# Upload the CSV file
uploaded = files.upload()

# Check if a file was uploaded
if uploaded:
    for fn in uploaded.keys():
        print(f'User uploaded file "{fn}" with length {len(uploaded[fn])} bytes')

        # Rename the uploaded file to california_housing.csv
        new_file_name = 'california_housing.csv'
        os.rename(fn, new_file_name)
        print(f'File renamed to "{new_file_name}"')
else:
    print("No file was uploaded.")

Saving housing[1].csv to housing[1].csv
User uploaded file "housing[1].csv" with length 1423529 bytes
File renamed to "california_housing.csv"


In [ ]:
K = 8
BATCH_SIZE = 256
N_BLOCKS = 3
D_BLOCK = 128
DROPOUT = 0.1
LR_MLP = 1e-3
LR_TABM = 2e-3
WEIGHT_DECAY = 0.0
MAX_EPOCHS = 100
PATIENCE = 16

In [ ]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import QuantileTransformer
import time

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
torch.manual_seed(0)

df = pd.read_csv('california_housing.csv')
import numpy as np
import pandas as pd

df = pd.read_csv('california_housing.csv')

# ---------------------------
# 1. Inspect before touching anything
# ---------------------------
print("Shape:", df.shape)
print("\nColumn dtypes:\n", df.dtypes)
print("\nMissing values per column:\n", df.isna().sum())
print("\nDuplicate rows:", df.duplicated().sum())

TARGET = 'median_house_value'  # <-- confirm this matches your actual column name

# ---------------------------
# 2. Drop rows with missing target (can't train on unlabeled rows)
# ---------------------------
before = len(df)
df = df.dropna(subset=[TARGET])
print(f"\nDropped {before - len(df)} rows with missing target")

# ---------------------------
# 3. Drop exact duplicate rows
# ---------------------------
before = len(df)
df = df.drop_duplicates()
print(f"Dropped {before - len(df)} duplicate rows")

# ---------------------------
# 4. Separate numeric and categorical columns
# ---------------------------
feature_cols = [c for c in df.columns if c != TARGET]
numeric_cols = df[feature_cols].select_dtypes(include=[np.number]).columns.tolist()
categorical_cols = df[feature_cols].select_dtypes(exclude=[np.number]).columns.tolist()
print(f"\nNumeric features ({len(numeric_cols)}): {numeric_cols}")
print(f"Categorical features ({len(categorical_cols)}): {categorical_cols}")

# ---------------------------
# 5. Impute missing numeric values (median is robust to outliers/skew)
# ---------------------------
missing_before = df[numeric_cols].isna().sum().sum()
for col in numeric_cols:
    if df[col].isna().any():
        median_val = df[col].median()
        df[col] = df[col].fillna(median_val)
        print(f"  Filled {col}: {df[col].isna().sum()} NaNs remaining (used median={median_val:.2f})")
print(f"Total numeric NaNs filled: {missing_before}")

# ---------------------------
# 6. Handle categorical missing values, then one-hot encode
# ---------------------------
for col in categorical_cols:
    if df[col].isna().any():
        df[col] = df[col].fillna(df[col].mode()[0])
df = pd.get_dummies(df, columns=categorical_cols)

# ---------------------------
# 7. Sanity-check for infinities (can silently appear from prior transforms/divide-by-zero features)
# ---------------------------
numeric_now = df.select_dtypes(include=[np.number]).columns
inf_counts = np.isinf(df[numeric_now]).sum()
if inf_counts.sum() > 0:
    print("\nWARNING — infinite values found:\n", inf_counts[inf_counts > 0])
    df[numeric_now] = df[numeric_now].replace([np.inf, -np.inf], np.nan)
    df[numeric_now] = df[numeric_now].fillna(df[numeric_now].median())

# ---------------------------
# 8. Final verification — this must print all zeros before you proceed
# ---------------------------
assert df.isna().sum().sum() == 0, "NaNs still present — do not proceed to training"
assert not np.isinf(df.select_dtypes(include=[np.number])).any().any(), "Infs still present"
print(f"\nFinal shape: {df.shape}")
print("Data is clean — safe to proceed to train/val/test split.")
print(df.columns.tolist())  # <-- CHECK this and fix the target name below

TARGET = 'median_house_value'  # <-- adjust if your CSV uses a different name
df = df.dropna(subset=[TARGET])

# If there's a categorical column like 'ocean_proximity', one-hot encode it;
# otherwise this line is a no-op.
df = pd.get_dummies(df, columns=[c for c in ['ocean_proximity'] if c in df.columns])

X = df.drop(columns=[TARGET]).values.astype(np.float32)
y = df[TARGET].values.astype(np.float32)

X_train, X_tmp, y_train, y_tmp = train_test_split(X, y, test_size=0.3, random_state=0)
X_val, X_test, y_val, y_test = train_test_split(X_tmp, y_tmp, test_size=0.5, random_state=0)

# Quantile normalization, as in the paper (fit on train only)
qt = QuantileTransformer(output_distribution='normal', random_state=0)
X_train = qt.fit_transform(X_train).astype(np.float32)
X_val = qt.transform(X_val).astype(np.float32)
X_test = qt.transform(X_test).astype(np.float32)

# Standardize target for stable training (report RMSE in original scale later)
y_mean, y_std = y_train.mean(), y_train.std()
y_train_n = (y_train - y_mean) / y_std
y_val_n = (y_val - y_mean) / y_std
y_test_n = (y_test - y_mean) / y_std

def to_tensor(a): return torch.tensor(a, device=device)
X_train_t, y_train_t = to_tensor(X_train), to_tensor(y_train_n)
X_val_t, y_val_t = to_tensor(X_val), to_tensor(y_val_n)
X_test_t, y_test_t = to_tensor(X_test), to_tensor(y_test_n)

n_features = X_train.shape[1]

def iterate_batches(X, y, batch_size, shuffle=True):
    n = X.shape[0]
    idx = torch.randperm(n, device=device) if shuffle else torch.arange(n, device=device)
    for i in range(0, n, batch_size):
        b = idx[i:i+batch_size]
        yield X[b], y[b]

def rmse_original_scale(pred_n, y_n):
    # undo the standardization before computing RMSE
    pred = pred_n * y_std + y_mean
    true = y_n * y_std + y_mean
    return torch.sqrt(F.mse_loss(pred, true)).item()

Shape: (20640, 10)

Column dtypes:
 longitude             float64
latitude              float64
housing_median_age    float64
total_rooms           float64
total_bedrooms        float64
population            float64
households            float64
median_income         float64
median_house_value    float64
ocean_proximity        object
dtype: object

Missing values per column:
 longitude               0
latitude                0
housing_median_age      0
total_rooms             0
total_bedrooms        207
population              0
households              0
median_income           0
median_house_value      0
ocean_proximity         0
dtype: int64

Duplicate rows: 0

Dropped 0 rows with missing target
Dropped 0 duplicate rows

Numeric features (8): ['longitude', 'latitude', 'housing_median_age', 'total_rooms', 'total_bedrooms', 'population', 'households', 'median_income']
Categorical features (1): ['ocean_proximity']
  Filled total_bedrooms: 0 NaNs remaining (used median=435.00)
Total nume

**1) Plain MLP :**

In [ ]:
class MLP(nn.Module):
    def __init__(self, n_features, d_block, n_blocks, dropout):
        super().__init__()
        layers = []
        d_in = n_features
        for _ in range(n_blocks):
            layers += [nn.Linear(d_in, d_block), nn.ReLU(), nn.Dropout(dropout)]
            d_in = d_block
        self.backbone = nn.Sequential(*layers)
        self.head = nn.Linear(d_in, 1)

    def forward(self, x):
        return self.head(self.backbone(x)).squeeze(-1)

def train_mlp(seed, X_train_t, y_train_t, X_val_t, y_val_t, verbose=False):
    torch.manual_seed(seed)
    model = MLP(n_features, D_BLOCK, N_BLOCKS, DROPOUT).to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=LR_MLP, weight_decay=WEIGHT_DECAY)

    best_val = float('inf')
    best_state = None
    patience_left = PATIENCE

    for epoch in range(MAX_EPOCHS):
        model.train()
        for xb, yb in iterate_batches(X_train_t, y_train_t, BATCH_SIZE):
            opt.zero_grad()
            loss = F.mse_loss(model(xb), yb)
            loss.backward()
            opt.step()

        model.eval()
        with torch.no_grad():
            val_pred = model(X_val_t)
            val_loss = F.mse_loss(val_pred, y_val_t).item()

        if val_loss < best_val:
            best_val = val_loss
            best_state = {k: v.clone() for k, v in model.state_dict().items()}
            patience_left = PATIENCE
        else:
            patience_left -= 1
            if patience_left == 0:
                break
        if verbose:
            print(f"epoch {epoch}: val_loss={val_loss:.4f}")

    model.load_state_dict(best_state)
    return model

t0 = time.time()
mlp_single = train_mlp(seed=0, X_train_t=X_train_t, y_train_t=y_train_t,
                        X_val_t=X_val_t, y_val_t=y_val_t)
train_time_mlp = time.time() - t0

mlp_single.eval()
with torch.no_grad():
    rmse_mlp = rmse_original_scale(mlp_single(X_test_t), y_test_t)
print(f"MLP  RMSE={rmse_mlp:.4f}  train_time={train_time_mlp:.1f}s")

MLP  RMSE=50306.2461  train_time=16.1s


**2 MLPxK :**

In [ ]:
t0 = time.time()
mlp_ensemble = [
    train_mlp(seed=s, X_train_t=X_train_t, y_train_t=y_train_t,
              X_val_t=X_val_t, y_val_t=y_val_t)
    for s in range(K)
]
train_time_mlpk = time.time() - t0

with torch.no_grad():
    preds = torch.stack([m(X_test_t) for m in mlp_ensemble], dim=0)  # (K, N)
    pred_mean = preds.mean(dim=0)
    rmse_mlpk = rmse_original_scale(pred_mean, y_test_t)
print(f"MLPx{K} RMSE={rmse_mlpk:.4f}  train_time={train_time_mlpk:.1f}s")

MLPx8 RMSE=48690.9961  train_time=84.3s


**3)  TabM naive :**

In [ ]:
class LinearBatchEnsemble(nn.Module):
    """Shared W; per-submodel R (input-side), S (output-side), B (bias).
       l_i(x) = (x * R_i) @ W * S_i + B_i"""
    def __init__(self, in_features, out_features, k, scaling_init='random-signs'):
        super().__init__()
        self.W = nn.Linear(in_features, out_features, bias=False)
        self.R = nn.Parameter(torch.empty(k, in_features))
        self.S = nn.Parameter(torch.empty(k, out_features))
        self.B = nn.Parameter(torch.zeros(k, out_features))
        self._init_scaling(self.R, scaling_init)
        self._init_scaling(self.S, scaling_init)

    def _init_scaling(self, param, mode):
        with torch.no_grad():
            if mode == 'random-signs':
                param.copy_((torch.randint(0, 2, param.shape).float() * 2 - 1))
            elif mode == 'normal':
                param.normal_(mean=0.0, std=1.0)
            else:
                raise ValueError(mode)

    def forward(self, x):
        # x: (B, k, in_features)
        x = x * self.R
        x = self.W(x)
        return x * self.S + self.B


class LinearEnsemble(nn.Module):
    """Fully independent linear layer per submodel — used as TabM's prediction head."""
    def __init__(self, in_features, out_features, k):
        super().__init__()
        self.weight = nn.Parameter(torch.empty(k, out_features, in_features))
        self.bias = nn.Parameter(torch.zeros(k, out_features))
        for i in range(k):
            nn.init.kaiming_uniform_(self.weight[i], a=5 ** 0.5)

    def forward(self, x):
        # x: (B, k, in_features) -> (B, k, out_features)
        return torch.einsum('bki,koi->bko', x, self.weight) + self.bias


class TabMNaive(nn.Module):
    def __init__(self, n_features, d_block, n_blocks, k, dropout):
        super().__init__()
        self.k = k
        layers = []
        d_in = n_features
        for _ in range(n_blocks):
            layers += [LinearBatchEnsemble(d_in, d_block, k), nn.ReLU(), nn.Dropout(dropout)]
            d_in = d_block
        self.backbone = nn.Sequential(*layers)
        self.head = LinearEnsemble(d_in, 1, k)

    def forward(self, x):
        # x: (B, n_features) -> k identical views -> (B, k, n_features)
        x = x.unsqueeze(1).expand(-1, self.k, -1)
        x = self.backbone(x)
        x = self.head(x).squeeze(-1)  # (B, k)
        return x


def train_tabm_naive(seed, X_train_t, y_train_t, X_val_t, y_val_t, verbose=False):
    torch.manual_seed(seed)
    model = TabMNaive(n_features, D_BLOCK, N_BLOCKS, K, DROPOUT).to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=LR_TABM, weight_decay=WEIGHT_DECAY)

    best_val = float('inf')
    best_state = None
    patience_left = PATIENCE

    for epoch in range(MAX_EPOCHS):
        model.train()
        for xb, yb in iterate_batches(X_train_t, y_train_t, BATCH_SIZE):
            opt.zero_grad()
            pred = model(xb)                       # (B, k)
            yb_expanded = yb.unsqueeze(1).expand(-1, K)  # (B, k)
            # CRITICAL: mean of per-submodel losses, NOT loss of the mean prediction
            loss = F.mse_loss(pred, yb_expanded)
            loss.backward()
            opt.step()

        model.eval()
        with torch.no_grad():
            val_pred = model(X_val_t).mean(dim=1)   # average for validation metric
            val_loss = F.mse_loss(val_pred, y_val_t).item()

        if val_loss < best_val:
            best_val = val_loss
            best_state = {k_: v.clone() for k_, v in model.state_dict().items()}
            patience_left = PATIENCE
        else:
            patience_left -= 1
            if patience_left == 0:
                break
        if verbose:
            print(f"epoch {epoch}: val_loss={val_loss:.4f}")

    model.load_state_dict(best_state)
    return model

t0 = time.time()
tabm_naive = train_tabm_naive(seed=0, X_train_t=X_train_t, y_train_t=y_train_t,
                               X_val_t=X_val_t, y_val_t=y_val_t)
train_time_tabm = time.time() - t0

tabm_naive.eval()
with torch.no_grad():
    pred = tabm_naive(X_test_t).mean(dim=1)  # average k predictions at inference
    rmse_tabm = rmse_original_scale(pred, y_test_t)
print(f"TabM_naive RMSE={rmse_tabm:.4f}  train_time={train_time_tabm:.1f}s")

TabM_naive RMSE=48536.8633  train_time=15.8s


In [ ]:
def run_multi_seed(train_fn, n_seeds=5, is_tabm=False):
    rmses = []
    for seed in range(n_seeds):
        model = train_fn(seed=seed, X_train_t=X_train_t, y_train_t=y_train_t,
                          X_val_t=X_val_t, y_val_t=y_val_t)
        model.eval()
        with torch.no_grad():
            pred = model(X_test_t)
            if is_tabm:
                pred = pred.mean(dim=1)
            rmses.append(rmse_original_scale(pred, y_test_t))
    return np.mean(rmses), np.std(rmses)

mlp_mean, mlp_std = run_multi_seed(train_mlp, n_seeds=5)
tabm_mean, tabm_std = run_multi_seed(train_tabm_naive, n_seeds=5, is_tabm=True)
print(f"MLP:        {mlp_mean:.1f} ± {mlp_std:.1f}")
print(f"TabM_naive: {tabm_mean:.1f} ± {tabm_std:.1f}")

MLP:        50100.9 ± 337.8
TabM_naive: 48713.0 ± 243.2


**4)TabM packed**

In [ ]:
class TabMPacked(nn.Module):
    """k independent MLPs, batched together. Every layer uses LinearEnsemble
       (fully independent per-submodel weights) — nothing is shared, unlike
       TabM_naive/TabM which use LinearBatchEnsemble (shared W)."""
    def __init__(self, n_features, d_block, n_blocks, k, dropout):
        super().__init__()
        self.k = k
        layers = []
        d_in = n_features
        for _ in range(n_blocks):
            layers += [LinearEnsemble(d_in, d_block, k), nn.ReLU(), nn.Dropout(dropout)]
            d_in = d_block
        self.backbone = nn.Sequential(*layers)
        self.head = LinearEnsemble(d_in, 1, k)

    def forward(self, x):
        # x: (B, n_features) -> k independent input copies -> (B, k, n_features)
        x = x.unsqueeze(1).expand(-1, self.k, -1)
        x = self.backbone(x)
        return self.head(x).squeeze(-1)  # (B, k)


def train_tabm_packed(seed, X_train_t, y_train_t, X_val_t, y_val_t, verbose=False):
    torch.manual_seed(seed)
    model = TabMPacked(n_features, D_BLOCK, N_BLOCKS, K, DROPOUT).to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=LR_TABM, weight_decay=WEIGHT_DECAY)

    best_val = float('inf')
    best_state = None
    patience_left = PATIENCE

    for epoch in range(MAX_EPOCHS):
        model.train()
        for xb, yb in iterate_batches(X_train_t, y_train_t, BATCH_SIZE):
            opt.zero_grad()
            pred = model(xb)                              # (B, k)
            yb_expanded = yb.unsqueeze(1).expand(-1, K)    # (B, k)
            loss = F.mse_loss(pred, yb_expanded)           # mean of per-submodel losses
            loss.backward()
            opt.step()

        model.eval()
        with torch.no_grad():
            val_pred = model(X_val_t).mean(dim=1)          # ensemble-aware stopping
            val_loss = F.mse_loss(val_pred, y_val_t).item()

        if val_loss < best_val:
            best_val = val_loss
            best_state = {k_: v.clone() for k_, v in model.state_dict().items()}
            patience_left = PATIENCE
        else:
            patience_left -= 1
            if patience_left == 0:
                break
        if verbose:
            print(f"epoch {epoch}: val_loss={val_loss:.4f}")

    if best_state is None:
        raise RuntimeError("Training never improved — check for NaNs.")
    model.load_state_dict(best_state)
    return model


t0 = time.time()
tabm_packed = train_tabm_packed(seed=0, X_train_t=X_train_t, y_train_t=y_train_t,
                                 X_val_t=X_val_t, y_val_t=y_val_t)
train_time_tabm_packed = time.time() - t0

tabm_packed.eval()
with torch.no_grad():
    pred = tabm_packed(X_test_t).mean(dim=1)  # average k predictions at inference
    rmse_tabm_packed = rmse_original_scale(pred, y_test_t)
print(f"TabM_packed RMSE={rmse_tabm_packed:.4f}  train_time={train_time_tabm_packed:.1f}s")

# 5-seed run — matches the multi-seed protocol used for TabM_naive/TabM_mini
packed_mean, packed_std = run_multi_seed(train_tabm_packed, n_seeds=5, is_tabm=True)
print(f"TabM_packed (5-seed): {packed_mean:.4f} ± {packed_std:.4f}")

# Sanity check: params should be ~K× an MLP's (no sharing yet) — the drop
# only shows up once TabM_naive introduces LinearBatchEnsemble
n_params_packed = sum(p.numel() for p in tabm_packed.parameters())
n_params_mlp = sum(p.numel() for p in mlp_single.parameters()) if 'mlp_single' in globals() else None
print(f"TabM_packed params: {n_params_packed:,}" +
      (f"  (single MLP: {n_params_mlp:,}, ~{n_params_packed / n_params_mlp:.1f}x)" if n_params_mlp else ""))

TabM_packed RMSE=48208.7070  train_time=14.5s
TabM_packed (5-seed): 48362.7531 ± 224.4621
TabM_packed params: 279,560  (single MLP: 34,945, ~8.0x)


**5) TabM with better initialization**

In [ ]:
class LinearBatchEnsembleV2(nn.Module):
    """Same as before, but R and S can now have independent init modes."""
    def __init__(self, in_features, out_features, k,
                 scaling_init_R='random-signs', scaling_init_S='random-signs'):
        super().__init__()
        self.W = nn.Linear(in_features, out_features, bias=False)
        self.R = nn.Parameter(torch.empty(k, in_features))
        self.S = nn.Parameter(torch.empty(k, out_features))
        self.B = nn.Parameter(torch.zeros(k, out_features))
        self._init_scaling(self.R, scaling_init_R)
        self._init_scaling(self.S, scaling_init_S)

    def _init_scaling(self, param, mode):
        with torch.no_grad():
            if mode == 'random-signs':
                param.copy_(torch.randint(0, 2, param.shape).float() * 2 - 1)
            elif mode == 'normal':
                param.normal_(mean=0.0, std=1.0)
            elif mode == 'ones':
                param.fill_(1.0)
            else:
                raise ValueError(mode)

    def forward(self, x):
        x = x * self.R
        x = self.W(x)
        return x * self.S + self.B


class TabM(nn.Module):
    """TabM = TabM_naive backbone + better initialization.
       Only the very first R stays random; everything else (S of layer 1,
       R and S of all later layers) starts at 1.0 and diverges during training."""
    def __init__(self, n_features, d_block, n_blocks, k, dropout, scaling_init='random-signs'):
        super().__init__()
        self.k = k
        layers = []
        d_in = n_features
        for i in range(n_blocks):
            if i == 0:
                layer = LinearBatchEnsembleV2(
                    d_in, d_block, k,
                    scaling_init_R=scaling_init,   # random — the one critical adapter
                    scaling_init_S='ones',         # deterministic at init
                )
            else:
                layer = LinearBatchEnsembleV2(
                    d_in, d_block, k,
                    scaling_init_R='ones',         # deterministic at init
                    scaling_init_S='ones',         # deterministic at init
                )
            layers += [layer, nn.ReLU(), nn.Dropout(dropout)]
            d_in = d_block
        self.backbone = nn.Sequential(*layers)
        self.head = LinearEnsemble(d_in, 1, k)  # still independent per submodel, as before

    def forward(self, x):
        x = x.unsqueeze(1).expand(-1, self.k, -1)  # (B, k, n_features)
        x = self.backbone(x)
        return self.head(x).squeeze(-1)  # (B, k)


def train_tabm(seed, X_train_t, y_train_t, X_val_t, y_val_t, verbose=False):
    torch.manual_seed(seed)
    model = TabM(n_features, D_BLOCK, N_BLOCKS, K, DROPOUT).to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=LR_TABM, weight_decay=WEIGHT_DECAY)

    best_val = float('inf')
    best_state = None
    patience_left = PATIENCE

    for epoch in range(MAX_EPOCHS):
        model.train()
        for xb, yb in iterate_batches(X_train_t, y_train_t, BATCH_SIZE):
            opt.zero_grad()
            pred = model(xb)                             # (B, k)
            yb_expanded = yb.unsqueeze(1).expand(-1, K)   # (B, k)
            loss = F.mse_loss(pred, yb_expanded)          # mean of per-submodel losses
            loss.backward()
            opt.step()

        model.eval()
        with torch.no_grad():
            val_pred = model(X_val_t).mean(dim=1)
            val_loss = F.mse_loss(val_pred, y_val_t).item()

        if val_loss < best_val:
            best_val = val_loss
            best_state = {k_: v.clone() for k_, v in model.state_dict().items()}
            patience_left = PATIENCE
        else:
            patience_left -= 1
            if patience_left == 0:
                break
        if verbose:
            print(f"epoch {epoch}: val_loss={val_loss:.4f}")

    if best_state is None:
        raise RuntimeError("Training never improved — val_loss was likely NaN throughout.")
    model.load_state_dict(best_state)
    return model


t0 = time.time()
tabm_model = train_tabm(seed=0, X_train_t=X_train_t, y_train_t=y_train_t,
                         X_val_t=X_val_t, y_val_t=y_val_t)
train_time_tabm_final = time.time() - t0

tabm_model.eval()
with torch.no_grad():
    pred = tabm_model(X_test_t).mean(dim=1)
    rmse_tabm_final = rmse_original_scale(pred, y_test_t)
print(f"TabM (better init) RMSE={rmse_tabm_final:.4f}  train_time={train_time_tabm_final:.1f}s")

TabM (better init) RMSE=48218.3086  train_time=15.7s


In [ ]:
torch.manual_seed(0)
check_model = TabM(n_features, D_BLOCK, N_BLOCKS, K, DROPOUT)
print("First layer R (should vary across rows):\n", check_model.backbone[0].R[:3])
print("First layer S (should be all 1.0):\n", check_model.backbone[0].S[:3])
print("Second layer R (should be all 1.0):\n", check_model.backbone[3].R[:3])

First layer R (should vary across rows):
 tensor([[-1.,  1., -1.,  1.,  1.,  1.,  1.,  1., -1., -1., -1., -1.,  1.],
        [ 1.,  1., -1., -1.,  1., -1., -1., -1.,  1., -1.,  1., -1.,  1.],
        [-1., -1.,  1.,  1.,  1., -1.,  1., -1.,  1., -1., -1., -1.,  1.]],
       grad_fn=<SliceBackward0>)
First layer S (should be all 1.0):
 tensor([[1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
         1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
         1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
         1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
         1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
         1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
         1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
         1., 1.],
        [1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1

In [ ]:
def train_tabm(seed, X_train_t, y_train_t, X_val_t, y_val_t, verbose=False):
    torch.manual_seed(seed)
    model = TabM(n_features, D_BLOCK, N_BLOCKS, K, DROPOUT).to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=LR_TABM, weight_decay=WEIGHT_DECAY)
    best_val = float('inf'); best_state = None; patience_left = PATIENCE
    epochs_run = 0

    for epoch in range(MAX_EPOCHS):
        epochs_run += 1
        model.train()
        for xb, yb in iterate_batches(X_train_t, y_train_t, BATCH_SIZE):
            opt.zero_grad()
            pred = model(xb)
            yb_expanded = yb.unsqueeze(1).expand(-1, K)
            loss = F.mse_loss(pred, yb_expanded)
            loss.backward()
            opt.step()
        model.eval()
        with torch.no_grad():
            val_loss = F.mse_loss(model(X_val_t).mean(dim=1), y_val_t).item()
        if val_loss < best_val:
            best_val = val_loss
            best_state = {k_: v.clone() for k_, v in model.state_dict().items()}
            patience_left = PATIENCE
        else:
            patience_left -= 1
            if patience_left == 0:
                break

    print(f"  seed={seed}: stopped after {epochs_run} epochs, best_val={best_val:.4f}")
    if best_state is None:
        raise RuntimeError("Training never improved.")
    model.load_state_dict(best_state)
    return model

In [ ]:
tabm_model = train_tabm(seed=0, X_train_t=X_train_t, y_train_t=y_train_t,
                         X_val_t=X_val_t, y_val_t=y_val_t, verbose=True)

tabm_model.eval()
with torch.no_grad():
    pred = tabm_model(X_test_t).mean(dim=1)
    rmse_tabm_final = rmse_original_scale(pred, y_test_t)
print(f"TabM RMSE={rmse_tabm_final:.4f}")

  seed=0: stopped after 100 epochs, best_val=0.1817
TabM RMSE=48218.3086


In [ ]:
print(device)  # should print cuda
print(torch.cuda.is_available())

cuda
True


In [ ]:
print(f"Device: {device}\n")

t0 = time.time()
mlp_final = train_mlp(seed=0, X_train_t=X_train_t, y_train_t=y_train_t, X_val_t=X_val_t, y_val_t=y_val_t)
t_mlp = time.time() - t0

t0 = time.time()
naive_final = train_tabm_naive(seed=0, X_train_t=X_train_t, y_train_t=y_train_t, X_val_t=X_val_t, y_val_t=y_val_t)
t_naive = time.time() - t0

t0 = time.time()
tabm_final = train_tabm(seed=0, X_train_t=X_train_t, y_train_t=y_train_t, X_val_t=X_val_t, y_val_t=y_val_t)
t_tabm = time.time() - t0

for name, m, is_tabm, t in [("MLP", mlp_final, False, t_mlp),
                             ("TabM_naive", naive_final, True, t_naive),
                             ("TabM (better init)", tabm_final, True, t_tabm)]:
    m.eval()
    with torch.no_grad():
        pred = m(X_test_t)
        if is_tabm: pred = pred.mean(dim=1)
        rmse = rmse_original_scale(pred, y_test_t)
    print(f"{name:<20} RMSE={rmse:>10.2f}  time={t:>6.1f}s")

Device: cuda

  seed=0: stopped after 100 epochs, best_val=0.1817
MLP                  RMSE=  50306.25  time=  10.7s
TabM_naive           RMSE=  48536.86  time=  15.8s
TabM (better init)   RMSE=  48218.31  time=  22.7s


In [ ]:
t0 = time.time()
mlp8_models = [train_mlp(seed=s, X_train_t=X_train_t, y_train_t=y_train_t,
                          X_val_t=X_val_t, y_val_t=y_val_t) for s in range(K)]
t_mlp8 = time.time() - t0

with torch.no_grad():
    preds = torch.stack([m(X_test_t) for m in mlp8_models], dim=0)
    rmse_mlp8 = rmse_original_scale(preds.mean(dim=0), y_test_t)
print(f"MLPx8                RMSE={rmse_mlp8:>10.2f}  time={t_mlp8:>6.1f}s")

MLPx8                RMSE=  48691.00  time=  85.1s


**6)TabM mini:**

In [ ]:
class ElementwiseAffine(nn.Module):
    """Per-submodel scaling applied once, at the input. This is the single R
       adapter that MiniEnsemble keeps — everything after this is fully shared."""
    def __init__(self, k, d_in, scaling_init='random-signs'):
        super().__init__()
        self.scale = nn.Parameter(torch.empty(k, d_in))
        with torch.no_grad():
            if scaling_init == 'random-signs':
                self.scale.copy_(torch.randint(0, 2, (k, d_in)).float() * 2 - 1)
            elif scaling_init == 'normal':
                self.scale.normal_(mean=0.0, std=1.0)
            else:
                raise ValueError(scaling_init)

    def forward(self, x):
        # x: (B, k, d_in)
        return x * self.scale


class TabMMini(nn.Module):
    def __init__(self, n_features, d_block, n_blocks, k, dropout, scaling_init='random-signs'):
        super().__init__()
        self.k = k
        self.affine = ElementwiseAffine(k, n_features, scaling_init)

        layers = []
        d_in = n_features
        for _ in range(n_blocks):
            # Ordinary nn.Linear — fully shared across all k submodels.
            # (nn.Linear applies to the last dim only, so it broadcasts
            # cleanly over the (B, k, d) shape without any special handling.)
            layers += [nn.Linear(d_in, d_block), nn.ReLU(), nn.Dropout(dropout)]
            d_in = d_block
        self.backbone = nn.Sequential(*layers)

        self.head = LinearEnsemble(d_in, 1, k)  # still independent per submodel, as before

    def forward(self, x):
        x = x.unsqueeze(1).expand(-1, self.k, -1)  # (B, k, n_features)
        x = self.affine(x)                          # only place submodels diverge before the shared backbone
        x = self.backbone(x)
        return self.head(x).squeeze(-1)              # (B, k)


def train_tabm_mini(seed, X_train_t, y_train_t, X_val_t, y_val_t, verbose=False):
    torch.manual_seed(seed)
    model = TabMMini(n_features, D_BLOCK, N_BLOCKS, K, DROPOUT).to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=LR_TABM, weight_decay=WEIGHT_DECAY)
    best_val = float('inf'); best_state = None; patience_left = PATIENCE
    epochs_run = 0

    for epoch in range(MAX_EPOCHS):
        epochs_run += 1
        model.train()
        for xb, yb in iterate_batches(X_train_t, y_train_t, BATCH_SIZE):
            opt.zero_grad()
            pred = model(xb)                              # (B, k)
            yb_expanded = yb.unsqueeze(1).expand(-1, K)
            loss = F.mse_loss(pred, yb_expanded)           # mean of per-submodel losses — same rule as before
            loss.backward()
            opt.step()

        model.eval()
        with torch.no_grad():
            val_loss = F.mse_loss(model(X_val_t).mean(dim=1), y_val_t).item()

        if val_loss < best_val:
            best_val = val_loss
            best_state = {k_: v.clone() for k_, v in model.state_dict().items()}
            patience_left = PATIENCE
        else:
            patience_left -= 1
            if patience_left == 0:
                break
        if verbose:
            print(f"epoch {epoch}: val_loss={val_loss:.4f}")

    print(f"  seed={seed}: stopped after {epochs_run} epochs, best_val={best_val:.4f}")
    if best_state is None:
        raise RuntimeError("Training never improved — check for NaNs.")
    model.load_state_dict(best_state)
    return model


t0 = time.time()
if torch.cuda.is_available():
    torch.cuda.synchronize()
tabm_mini = train_tabm_mini(seed=0, X_train_t=X_train_t, y_train_t=y_train_t,
                             X_val_t=X_val_t, y_val_t=y_val_t)
if torch.cuda.is_available():
    torch.cuda.synchronize()
t_mini = time.time() - t0

tabm_mini.eval()
with torch.no_grad():
    rmse_mini = rmse_original_scale(tabm_mini(X_test_t).mean(dim=1), y_test_t)
print(f"TabM_mini            RMSE={rmse_mini:>10.2f}  time={t_mini:>6.1f}s")

  seed=0: stopped after 100 epochs, best_val=0.1925
TabM_mini            RMSE=  49522.68  time=  13.3s


In [ ]:
mini_mean, mini_std = run_multi_seed(train_tabm_mini, n_seeds=5, is_tabm=True)
print(f"TabM_mini: {mini_mean:.1f} ± {mini_std:.1f}")

  seed=0: stopped after 100 epochs, best_val=0.1925
  seed=1: stopped after 100 epochs, best_val=0.1898
  seed=2: stopped after 100 epochs, best_val=0.1837
  seed=3: stopped after 100 epochs, best_val=0.1905
  seed=4: stopped after 100 epochs, best_val=0.1893
TabM_mini: 49259.0 ± 463.7


In [ ]:
class TabMMiniV2(TabMMini):
    def __init__(self, n_features, d_block, n_blocks, k, dropout):
        super().__init__(n_features, d_block, n_blocks, k, dropout, scaling_init='normal')

def train_tabm_mini_normal(seed, X_train_t, y_train_t, X_val_t, y_val_t, verbose=False):
    torch.manual_seed(seed)
    model = TabMMiniV2(n_features, D_BLOCK, N_BLOCKS, K, DROPOUT).to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=LR_TABM, weight_decay=WEIGHT_DECAY)
    best_val = float('inf')
    best_state = None
    patience_left = PATIENCE
    epochs_run = 0

    for epoch in range(MAX_EPOCHS):
        epochs_run += 1
        model.train()
        for xb, yb in iterate_batches(X_train_t, y_train_t, BATCH_SIZE):
            opt.zero_grad()
            pred = model(xb)
            yb_expanded = yb.unsqueeze(1).expand(-1, K)
            loss = F.mse_loss(pred, yb_expanded)
            loss.backward()
            opt.step()

        model.eval()
        with torch.no_grad():
            val_loss = F.mse_loss(model(X_val_t).mean(dim=1), y_val_t).item()

        if val_loss < best_val:
            best_val = val_loss
            best_state = {k_: v.clone() for k_, v in model.state_dict().items()}
            patience_left = PATIENCE
        else:
            patience_left -= 1
            if patience_left == 0:
                break

    if best_state is None:
        raise RuntimeError("Training never improved — check for NaNs.")
    model.load_state_dict(best_state)
    return model

mini_normal_mean, mini_normal_std = run_multi_seed(train_tabm_mini_normal, n_seeds=5, is_tabm=True)
print(f"TabM_mini (normal init): {mini_normal_mean:.1f} ± {mini_normal_std:.1f}")

TabM_mini (normal init): 49089.5 ± 438.7


# **Final combined results**

In [ ]:
from sklearn.metrics import r2_score

def r2_of(pred_n, y_n=y_test_t):
    pred = (pred_n * y_std + y_mean).cpu().numpy()
    true = (y_n * y_std + y_mean).cpu().numpy()
    return r2_score(true, pred)

with torch.no_grad():
    r2_mlp    = r2_of(mlp_single(X_test_t))
    r2_mlpk   = r2_of(torch.stack([m(X_test_t) for m in mlp_ensemble], dim=0).mean(dim=0))
    r2_naive  = r2_of(tabm_naive(X_test_t).mean(dim=1))
    r2_packed = r2_of(tabm_packed(X_test_t).mean(dim=1))
    r2_tabm   = r2_of(tabm_model(X_test_t).mean(dim=1))
    r2_mini   = r2_of(tabm_mini(X_test_t).mean(dim=1))

n_params_mlp    = sum(p.numel() for p in mlp_single.parameters())
n_params_mlpk   = n_params_mlp * K
n_params_naive  = sum(p.numel() for p in tabm_naive.parameters())
n_params_packed = sum(p.numel() for p in tabm_packed.parameters())
n_params_tabm   = sum(p.numel() for p in tabm_model.parameters())
n_params_mini   = sum(p.numel() for p in tabm_mini.parameters())

results_df = pd.DataFrame([
    {"Model": "Plain MLP",                      "RMSE": rmse_mlp,         "R2 (Accuracy)": r2_mlp,    "Training Time (s)": train_time_mlp,        "Params": n_params_mlp},
    {"Model": "MLPxK",                           "RMSE": rmse_mlpk,        "R2 (Accuracy)": r2_mlpk,   "Training Time (s)": train_time_mlpk,       "Params": n_params_mlpk},
    {"Model": "TabM naive",                      "RMSE": rmse_tabm,        "R2 (Accuracy)": r2_naive,  "Training Time (s)": train_time_tabm,       "Params": n_params_naive},
    {"Model": "TabM packed",                     "RMSE": rmse_tabm_packed, "R2 (Accuracy)": r2_packed, "Training Time (s)": train_time_tabm_packed,"Params": n_params_packed},
    {"Model": "TabM with better initialization", "RMSE": rmse_tabm_final,  "R2 (Accuracy)": r2_tabm,   "Training Time (s)": train_time_tabm_final, "Params": n_params_tabm},
    {"Model": "TabM mini",                       "RMSE": rmse_mini,        "R2 (Accuracy)": r2_mini,   "Training Time (s)": t_mini,                "Params": n_params_mini},
]).round({"RMSE": 2, "R2 (Accuracy)": 4, "Training Time (s)": 2})

results_df

,Model,RMSE,R2 (Accuracy),Training Time (s),Params
0,Plain MLP,50306.25,0.8105,16.13,34945
1,MLPxK,48691.00,0.8224,84.30,279560
2,TabM naive,48536.86,0.8236,15.76,43760
3,TabM packed,48208.71,0.8259,14.52,279560
4,TabM with better initialization,48218.31,0.8259,15.68,43760
5,TabM mini,49522.68,0.8163,13.31,35952
